In [ ]:
from google.colab import drive
drive.mount('/content/drive')



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# FOLDER where your models + tokenizer live
BASE_DIR = "/content/drive/MyDrive/Disaster"  # 👈 CHANGE this to the correct folder

TEXT_MODEL_PATH  = f"{BASE_DIR}/disaster_tweet_model.keras"
TOKENIZER_PATH   = f"{BASE_DIR}/tokenizer_disaster.pkl"

# You have both best_resnet50_model.h5 and .keras – we’ll use the .keras one
IMAGE_MODEL_PATH = f"{BASE_DIR}/best_resnet50_model.keras"
# If that errors, switch to:
# IMAGE_MODEL_PATH = f"{BASE_DIR}/resnet50_disaster_classifier.keras"


In [ ]:
import pickle
import numpy as np

import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing import image as keras_image

# SAME preprocess_input you used during training
from tensorflow.keras.applications.resnet50 import preprocess_input

from PIL import Image

print("TensorFlow version:", tf.__version__)


TensorFlow version: 2.19.0


In [ ]:
MAX_LEN = 40  # 👈 CHANGE if you used a different maxlen during training

# 1) Load tokenizer
with open(TOKENIZER_PATH, "rb") as f:
    tokenizer = pickle.load(f)
print("✅ Tokenizer loaded")

# 2) Load BiLSTM + GloVe tweet model
text_model = load_model(TEXT_MODEL_PATH)
print("✅ Text model loaded")
text_model.summary()



✅ Tokenizer loaded
✅ Text model loaded


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 40)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, 40, 100)   │  1,578,000 │ input_layer_2[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_2     │ (None, 40, 256)   │    234,496 │ embedding_2[0][0] │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 40, 1)     │        257 │ bidirectional_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ softmax_1 (Softmax) │ (None, 40, 1)     │          0 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply (Multiply) │ (None, 40, 256)   │          0 │ softmax_1[0][0],  │
│                     │                   │            │ bidirectional_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 38, 128)   │     98,432 │ bidirectional_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sum_1 (Sum)         │ (None, 256)       │          0 │ multiply[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 128)       │          0 │ conv1d[0][0]      │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 384)       │          0 │ sum_1[0][0],      │
│ (Concatenate)       │                   │            │ global_max_pooli… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 128)       │     49,280 │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 128)       │          0 │ dense_4[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 64)        │      8,256 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 64)        │          0 │ dense_5[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 1)         │         65 │ dropout_3[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 5,906,360 (22.53 MB)

 Trainable params: 1,968,786 (7.51 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 3,937,574 (15.02 MB)

In [ ]:
def clean_text_for_inference(text: str) -> str:
    """
    Apply the SAME cleaning you used in training.
    If you had a clean_text() function, copy its logic here.
    For now: simple lowercasing + strip.
    """
    return text.strip().lower()

def predict_disaster_from_text(caption: str, threshold: float = 0.5):
    cleaned = clean_text_for_inference(caption)
    seq = tokenizer.texts_to_sequences([cleaned])
    pad = pad_sequences(seq, maxlen=MAX_LEN, padding="post", truncating="post")

    prob_disaster = float(text_model.predict(pad, verbose=0)[0][0])

    is_disaster = prob_disaster >= threshold
    label = "disaster" if is_disaster else "not_disaster"

    return is_disaster, label, prob_disaster



In [ ]:
IMG_SIZE = (224, 224)

CLASS_NAMES = [
    "Damaged_Infrastructure",
    "Fire_Disaster",
    "Human_Damage",
    "Land_Disaster",
    "Non_Damage",
    "Water_Disaster"
]

print("✅ Using class order:", CLASS_NAMES)

# Try loading the .keras model first
try:
    image_model = load_model(IMAGE_MODEL_PATH)
    print("✅ Loaded image model:", IMAGE_MODEL_PATH)
except Exception as e:
    print("⚠️ Failed to load .keras model. Trying .h5 instead...")
    IMAGE_MODEL_PATH = IMAGE_MODEL_PATH.replace(".keras", ".h5")
    image_model = load_model(IMAGE_MODEL_PATH)
    print("✅ Loaded image model:", IMAGE_MODEL_PATH)

image_model.summary()


✅ Using class order: ['Damaged_Infrastructure', 'Fire_Disaster', 'Human_Damage', 'Land_Disaster', 'Non_Damage', 'Water_Disaster']
✅ Loaded image model: /content/drive/MyDrive/Disaster/best_resnet50_model.keras


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential          │ (None, 224, 224,  │          0 │ input_layer_1[0]… │
│ (Sequential)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item (GetItem)  │ (None, 224, 224)  │          0 │ sequential[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_1          │ (None, 224, 224)  │          0 │ sequential[0][0]  │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_2          │ (None, 224, 224)  │          0 │ sequential[0][0]  │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stack_1 (Stack)     │ (None, 224, 224,  │          0 │ get_item[0][0],   │
│                     │ 3)                │            │ get_item_1[0][0], │
│                     │                   │            │ get_item_2[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 224, 224,  │          0 │ stack_1[0][0]     │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet50            │ (None, 7, 7,      │ 23,587,712 │ add[0][0]         │
│ (Functional)        │ 2048)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 2048)      │          0 │ resnet50[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 2048)      │          0 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 6)         │     12,294 │ dropout[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 23,624,596 (90.12 MB)

 Trainable params: 12,294 (48.02 KB)

 Non-trainable params: 23,587,712 (89.98 MB)

 Optimizer params: 24,590 (96.06 KB)

In [ ]:
def load_and_preprocess_image(img_path: str):
    """
    Loads image and applies the SAME preprocessing as in training:
    - resize to (224, 224)
    - convert to array
    - preprocess_input (ResNet50)
    """
    img = keras_image.load_img(img_path, target_size=IMG_SIZE)
    img_array = keras_image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)  # (1, h, w, 3)
    img_array = preprocess_input(img_array)
    return img_array

def predict_disaster_type_from_image(image_path: str):
    """
    Returns:
        predicted_class (str)
        predicted_confidence (float)
    """
    img_batch = load_and_preprocess_image(image_path)
    probs = image_model.predict(img_batch, verbose=0)[0]  # shape: (num_classes,)

    idx = int(np.argmax(probs))
    pred_class = CLASS_NAMES[idx]
    pred_conf  = float(probs[idx])

    return pred_class, pred_conf


In [ ]:
def classify_post(caption: str, image_path: str, text_threshold: float = 0.5):
    """
    1. Run caption through BiLSTM+GloVe model (disaster / not_disaster).
    2. If NOT disaster → exit early, image model not used.
    3. If disaster     → run image through ResNet50 model to classify type.
    """
    result = {}

    # Step 1: Text model
    is_disaster, text_label, text_prob = predict_disaster_from_text(
        caption,
        threshold=text_threshold
    )

    result["caption"] = caption
    result["text_label"] = text_label
    result["text_disaster_prob"] = text_prob

    if not is_disaster:
        result["is_disaster"] = False
        result["image_used"] = False
        result["disaster_type"] = None
        result["image_confidence"] = None
        return result

    # Step 2: Image model
    disaster_type, img_conf = predict_disaster_type_from_image(image_path)

    result["is_disaster"] = True
    result["image_used"] = True
    result["disaster_type"] = disaster_type
    result["image_confidence"] = img_conf
    return result


In [ ]:
TEST_IMAGE_PATH = "/content/flood.jpeg"   # 👈 change to a real image in that folder
TEST_CAPTION    = "Huge flooding happening downtown, houses destroyed!"

result = classify_post(TEST_CAPTION, TEST_IMAGE_PATH, text_threshold=0.5)
result


{'caption': 'Huge flooding happening downtown, houses destroyed!',
 'text_label': 'disaster',
 'text_disaster_prob': 0.9893625974655151,
 'is_disaster': True,
 'image_used': True,
 'disaster_type': 'Water_Disaster',
 'image_confidence': 0.7051417231559753}